In [ ]:
import os
import json

In [ ]:
RUN_DIR = "../runs/d12m3"
assert os.path.exists(RUN_DIR), f"Run directory {RUN_DIR} does not exist."

In [ ]:
# list files
log_files = [fn for fn in os.listdir(RUN_DIR) if fn.startswith("train_log_") and fn.endswith(".jsonl")]
log_files = sorted(log_files)  # sort files to ensure rank order
print(log_files)

In [ ]:
logs = []
for log_file in log_files:
    with open(os.path.join(RUN_DIR, log_file), "r") as f:
        lines = f.readlines()
        logs.append(lines)
        print(f"{log_file}: {len(lines)} lines")

In [ ]:
logs[0][:5]

In [ ]:
logs_obj = [[json.loads(line) for line in log] for log in logs]

In [ ]:
logs_obj = [[entry for entry in log_list if entry['event'] == 'train'] for log_list in logs_obj]
for r in range(len(logs_obj)):
    print(f"Rank {r}: {len(logs_obj[r])} train entries")
num_steps = min([len(log_list) for log_list in logs_obj])
num_ranks = len(logs_obj)
print(f"Min steps: {num_steps}, Num ranks: {num_ranks}")


In [ ]:
logs_obj[0][0]

In [ ]:
merged_steps = []
for step in range(num_steps):
    step_merged = {}
    all_keys_all_ranks = list({k for r in range(num_ranks) for k in logs_obj[r][step].keys()})
    keys_sorted = sorted(all_keys_all_ranks)
    for k in keys_sorted:
        step_merged[k] = [logs_obj[r][step].get(k, None) for r in range(num_ranks)]
    merged_steps.append(step_merged)

In [ ]:
for k, v in merged_steps[0].items():
    print(f"{k}: {v}")

In [ ]:
merged_steps_2 = []
for step in merged_steps:
    step_merged = {}
    for k, v in step.items():
        if k == "step":
            assert all(v[i] == v[0] for i in range(len(v))), "Step values differ across ranks"
            step_merged["step"] = v[0]
        elif k == "event":
            assert all(v[i] == v[0] for i in range(len(v))), "Event values differ across ranks"
            step_merged["event"] = v[0]
        elif k == "timestamp":
            step_merged["timestamp"] = v
        elif k.startswith("train/") or k.startswith("other/"):
            step_merged[k] = v
        elif k.startswith("gpt/"):
            if k.startswith("gpt/transformer.wte"):
                step_merged[k] = v
            elif k.startswith("gpt/transformer"):
                k_parts = k.split(".")
                assert k_parts[1] == "h"
                assert len(k_parts) > 3
                block_id = int(k_parts[2])
                param_name = ".".join(k_parts[3:])
                param_name = param_name.replace("_grad_", ".grad__")
                param_name = param_name.replace("_params_", ".params__")
                param_name = param_name.replace("_update_", ".params__upd_")
                param_name = param_name.replace("x_fwd_", "resid.x_fwd__")
                param_trunk, sub_name = param_name.split("__", 1)

                if 'blocks' not in step_merged:
                    step_merged['blocks'] = {}
                if block_id not in step_merged['blocks']:
                    step_merged['blocks'][block_id] = {
                        'step': step['step'][0],
                        'block_id': block_id,
                    }
                if param_trunk not in step_merged['blocks'][block_id]:
                    step_merged['blocks'][block_id][param_trunk] = {}
                step_merged['blocks'][block_id][param_trunk][sub_name] = v
            else:
                step_merged[k] = v
    merged_steps_2.append(step_merged)

In [ ]:
for k, v in merged_steps_2[20]['blocks'][11].items():
    print(f"{k}: {v}")

In [ ]:
# Reduce
# Muon - sum across ranks, replace None with 0.0 for missing values
# Adam large/small - for _param_ and _update_ keys, check corresponding _is_small key
# Adam large - sum across ranks, there should be no None values
# Adam small - should be identical across ranks, take first non-None value

In [ ]:
block_list = []
for step in merged_steps_2:
    for block in step['blocks'].values():
        block_list.append(block)

In [ ]:
for k, v in block_list[1].items():
    print(f"{k}: {v}")

In [ ]:
# Remove ve_gate keys
for rb in block_list:
    for k in list(rb.keys()):
        if 've_gate' in k:
            del rb[k]

In [ ]:
"""
step: 0
block_id: 1
attn.c_k.weight.grad: {'num_el': [589824, 589824], 'sq_sum': [0.0, 0.0]}
attn.c_k.weight.params: {'is_small': [False, None], 'num_el': [589824, None], 'sq_sum': [769.6942138671875, None], 'upd_sq_sum': [1.5086006897035986e-05, None]}
attn.c_proj.weight.grad: {'num_el': [589824, 589824], 'sq_sum': [3.052793090319028e-06, 3.539155386533821e-06]}
attn.c_proj.weight.params: {'is_small': [False, None], 'num_el': [589824, None], 'sq_sum': [0.0, None], 'upd_sq_sum': [0.0001471763534937054, None]}
attn.c_q.weight.grad: {'num_el': [589824, 589824], 'sq_sum': [0.0, 0.0]}
attn.c_q.weight.params: {'is_small': [False, None], 'num_el': [589824, None], 'sq_sum': [769.730224609375, None], 'upd_sq_sum': [1.5086712664924562e-05, None]}
attn.c_v.weight.grad: {'num_el': [589824, 589824], 'sq_sum': [0.0, 0.0]}
attn.c_v.weight.params: {'is_small': [False, None], 'num_el': [589824, None], 'sq_sum': [769.4988403320312, None], 'upd_sq_sum': [1.5082178833836224e-05, None]}
attn.ve_gate.weight.grad: {'num_el': [72, 72], 'sq_sum': [0.0, 0.0]}
attn.ve_gate.weight.params: {'is_small': [False, None], 'num_el': [72, None], 'sq_sum': [0.008725840598344803, None], 'upd_sq_sum': [1.7102648464106807e-10, None]}
mlp.c_fc.weight.grad: {'num_el': [2359296, 2359296], 'sq_sum': [0.0, 0.0]}
mlp.c_fc.weight.params: {'is_small': [False, None], 'num_el': [2359296, None], 'sq_sum': [490.84283447265625, None], 'upd_sq_sum': [3.8482081436086446e-05, None]}
mlp.c_proj.weight.grad: {'num_el': [2359296, 2359296], 'sq_sum': [1.5035071555757895e-05, 1.5116766007849947e-05]}
mlp.c_proj.weight.params: {'is_small': [False, None], 'num_el': [2359296, None], 'sq_sum': [0.0, None], 'upd_sq_sum': [0.00018643567455001175, None]}
resid.x_fwd: {'num_el': [201326592, 201326592], 'sq_sum': [600183344.0, 600192968.0]}
"""

def first_not_none(lst):
    for item in lst:
        if item is not None:
            return item
    assert False

def sum_not_none(lst):
    return sum(item for item in lst if item is not None)

reduced_blocks_params = []
for block in block_list:
    for k, v in block.items():
        if k in ['step', 'block_id']:
            pass
        elif k.endswith('.x_fwd'):
            num_el = sum(v['num_el'])
            sq_sum = sum(v['sq_sum'])
            rms = (sq_sum / num_el) ** 0.5
            reduced_block = {
                'step': block['step'],
                'block_id': block['block_id'],
                'name': f"{k}_rms",
                'value': rms,
            }
            reduced_blocks_params.append(reduced_block)
        elif k.endswith('.grad'):
            num_el = sum(v['num_el'])
            sq_sum = sum(v['sq_sum'])
            rms = (sq_sum / num_el) ** 0.5
            reduced_block = {
                'step': block['step'],
                'block_id': block['block_id'],
                'name': f"{k}_rms",
                'value': rms,
            }
            reduced_blocks_params.append(reduced_block)
        elif k.endswith('.params'):
            is_small = first_not_none(v['is_small'])
            if is_small:
                # should be identical across ranks, take first non-None value
                num_el = first_not_none(v['num_el'])
                sq_sum = first_not_none(v['sq_sum'])
                upd_sq_sum = first_not_none(v['upd_sq_sum'])
            else:
                num_el = sum_not_none(v['num_el'])
                sq_sum = sum_not_none(v['sq_sum'])
                upd_sq_sum = sum_not_none(v['upd_sq_sum'])
            update_rms = (upd_sq_sum / num_el) ** 0.5
            reduced_block = {
                'step': block['step'],
                'block_id': block['block_id'],
                'name': f"{k}_update_rms",
                'value': update_rms,
            }
            update_param_ratio = (upd_sq_sum / sq_sum) ** 0.5 if sq_sum > 0 else 0.0
            reduced_block = {
                'step': block['step'],
                'block_id': block['block_id'],
                'name': f"{k}_update_param_ratio",
                'value': update_param_ratio,
            }
            reduced_blocks_params.append(reduced_block)
        else:
            raise NotImplementedError(f"Reduction for key {k} not implemented yet")
        

In [ ]:
reduced_blocks_params[223]

In [ ]:
all_param_names = sorted(set(rb['name'] for rb in reduced_blocks_params))
for pn in all_param_names:
    print(pn)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
for name in all_param_names:
    print(name)

    plt.figure(figsize=(12,4))
    steps = list(range(len(merged_steps_2)))
    for b_idx in range(12):
        fwd_rms = [b['value'] for b in reduced_blocks_params if b['name'] == name and b['block_id'] == b_idx]
        plt.plot(steps, fwd_rms, label=f"{name} block {b_idx}")
    plt.title(name)
    plt.legend()
    plt.show()